[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/14_diffusion_flow_targets.ipynb)

# 14. Diffusion and flow training targets

모델 없이 x0, noise, time만 두고 epsilon/x0/v/flow/rectified-flow/mean-flow 계열 target tensor가 어떻게 만들어지는지 본다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. Forward noising

x0와 noise를 time-dependent coefficient로 섞는다.


In [ ]:
x0 = torch.tensor([[1., -1.]], device=device)
eps = torch.tensor([[0.5, 2.0]], device=device)
alpha = torch.tensor([[0.8]], device=device)
sigma = torch.sqrt(1 - alpha.square())

xt = alpha * x0 + sigma * eps

print("x0:", x0)
print("eps:", eps)
print("xt:", xt)


In [ ]:
_ = profile_call("forward noising", lambda: alpha * x0 + sigma * eps)


## 2. epsilon / x0 prediction targets

같은 xt를 놓고 target 정의만 바꾼다.


In [ ]:
target_eps = eps
target_x0 = x0

print("epsilon target:", target_eps)
print("x0 target:", target_x0)


In [ ]:
_ = profile_call("epsilon target", lambda: eps + 0)
_ = profile_call("x0 target", lambda: x0 + 0)


## 3. v-prediction

rotation-like combination으로 velocity target을 만든다.


In [ ]:
v = alpha * eps - sigma * x0
print("v target:", v)


In [ ]:
_ = profile_call("v target", lambda: alpha * eps - sigma * x0)


## 4. Linear flow matching

두 endpoint 사이의 직선 interpolation과 constant velocity를 만든다.


In [ ]:
z0 = torch.tensor([[0., 0.]], device=device)
z1 = torch.tensor([[2., -1.]], device=device)
t = torch.tensor([[0.25]], device=device)

zt = (1 - t) * z0 + t * z1
target_v = z1 - z0

print("zt:", zt)
print("velocity:", target_v)


In [ ]:
_ = profile_call("linear flow target", lambda: ((1-t)*z0 + t*z1, z1-z0))


## 5. Rectified-flow view

여러 t에서 같은 endpoint pair의 straight velocity를 확인한다.


In [ ]:
for t_value in [0.0, 0.25, 0.5, 0.75, 1.0]:
    t = torch.tensor([[t_value]], device=device)
    zt = (1 - t) * z0 + t * z1
    print(t_value, "zt", zt.tolist(), "v", target_v.tolist())


## 6. Mean-flow interval average

analytic field에서 구간 평균 velocity를 직접 적분해 instantaneous velocity와 비교한다.


In [ ]:
def field(x, t):
    return -x + t

x = torch.tensor([[2.0]], device=device)
t0, t1 = 0.2, 0.8
grid = torch.linspace(t0, t1, 9, device=device)

values = torch.stack([field(x, ti) for ti in grid], dim=0)
mean_v = values.mean(dim=0)

print("instantaneous at t0:", field(x, torch.tensor(t0, device=device)))
print("interval mean:", mean_v)


In [ ]:
_ = profile_call("mean field samples", lambda: torch.stack([field(x, ti) for ti in grid], 0).mean(0))


## References and provenance

**[14.1] DDPM targets**
- 출처: Ho et al., Denoising Diffusion Probabilistic Models
- 이 노트북에서 가져온 부분: epsilon prediction

**[14.2] v prediction**
- 출처: progressive distillation / Stable Diffusion-family parameterizations
- 이 노트북에서 가져온 부분: velocity parameterization

**[14.3] Flow Matching**
- 출처: Lipman et al., Flow Matching for Generative Modeling
- 이 노트북에서 가져온 부분: conditional vector-field target

**[14.4] Rectified Flow**
- 출처: Liu et al., Flow Straight and Fast
- 이 노트북에서 가져온 부분: straight transport / reflow viewpoint

**[14.5] MeanFlow**
- 출처: MeanFlow paper
- 이 노트북에서 가져온 부분: interval-averaged velocity objective
